# Import Library

In [1]:
import pandas as pd

# Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load the dataset

In [3]:
file_path = '/content/drive/MyDrive/FYP Data Cleaning/Raw Data/personal_transactions_dashboard_ready (2).csv'
df1 = pd.read_csv(file_path)

# Display initial shape and info of the dataset

In [4]:
print("--- Initial Dataset Shape ---")
print(df1.shape)

--- Initial Dataset Shape ---
(806, 7)


In [5]:
print("\n--- Initial Info & Data Types ---")
print(df1.info())


--- Initial Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806 entries, 0 to 805
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              806 non-null    object 
 1   Description       806 non-null    object 
 2   Amount            806 non-null    float64
 3   Transaction Type  806 non-null    object 
 4   Category          806 non-null    object 
 5   Account Name      806 non-null    object 
 6   Month             806 non-null    object 
dtypes: float64(1), object(6)
memory usage: 44.2+ KB
None


:# Check for missing values

In [7]:
print(df1.isnull().sum())

Date                0
Description         0
Amount              0
Transaction Type    0
Category            0
Account Name        0
Month               0
dtype: int64


# Check for duplicate rows

In [ ]:
print("\n--- Check for Duplicate Rows ---")
print(f"Number of duplicate rows: {df1.duplicated().sum()}")


--- Check for Duplicate Rows ---
Number of duplicate rows: 0


# Count of duplicates

In [8]:
dup_rows = df1.duplicated().sum()

print("--- Duplicate Check (before cleaning) ---")
print(f"Total rows                   : {len(df1)}")
print(f"Duplicate rows (all columns) : {dup_rows}")

--- Duplicate Check (before cleaning) ---
Total rows                   : 806
Duplicate rows (all columns) : 0


# Handle missing values

In [9]:
before = len(df1)
df1 = df1.dropna(subset=['Date', 'Amount'])
print(f"Rows dropped (missing Date/Amount): {before - len(df1)}")

Rows dropped (missing Date/Amount): 0


# Remove any duplicate

In [10]:
before = len(df1)
df1 = df1.drop_duplicates().reset_index(drop=True)
print(f"Removed   : {before - len(df1)}")
print(f"Remaining : {len(df1)}")

Removed   : 0
Remaining : 806


# Standardize data types

In [11]:
df1['Date'] = pd.to_datetime(df1['Date'], errors='coerce')
print("Invalid dates (NaT):", df1['Date'].isna().sum())

Invalid dates (NaT): 0


# Convert Amount to LKR

In [12]:
ASSUMED_CURRENCY = 'USD'

# Exchange rates: 1 unit of currency = ? LKR
RATE_DATE   = "2026-10-14"
RATE_SOURCE = "Central Bank of Sri Lanka (daily indicative rates)"

RATES_TO_LKR = {
    'USD': 331.0482,
    'EUR': 370.9726,
    'GBP': 439.2016,
    'JPY': 2.0902,
    'INR': 3.4744,
    'AUD': 231.5020,
    'CAD': 233.0341,
    'LKR': 1.0,
}

if ASSUMED_CURRENCY not in RATES_TO_LKR:
    raise ValueError(f"No rate available for {ASSUMED_CURRENCY}. Add it to RATES_TO_LKR.")

rate = RATES_TO_LKR[ASSUMED_CURRENCY]

# Add new columns; the original 'Amount' column stays unchanged
df1['Original_Currency'] = ASSUMED_CURRENCY
df1['Amount_LKR'] = (df1['Amount'] * rate).round(2)

print(f"Assumed currency: {ASSUMED_CURRENCY} | Rate: {rate} | {RATE_SOURCE}, {RATE_DATE}")
print(df1[['Amount', 'Original_Currency', 'Amount_LKR']].head(10))
print("Missing Amount_LKR:", df1['Amount_LKR'].isna().sum())

Assumed currency: USD | Rate: 331.0482 | Central Bank of Sri Lanka (daily indicative rates), 2026-10-14
    Amount Original_Currency  Amount_LKR
0    11.11               USD     3677.95
1  1247.44               USD   412962.77
2    24.22               USD     8017.99
3  2298.09               USD   760778.56
4    11.76               USD     3893.13
5    25.85               USD     8557.60
6    18.45               USD     6107.84
7    45.00               USD    14897.17
8    15.38               USD     5091.52
9    10.69               USD     3538.91
Missing Amount_LKR: 0


# Verify final shape and view sample data

In [13]:
print("\n--- Cleaned Dataset Shape ---")
print(df1.shape)


--- Cleaned Dataset Shape ---
(806, 9)


In [14]:
print("\n--- Remaining Missing Values ---")
print(df1.isnull().sum())


--- Remaining Missing Values ---
Date                 0
Description          0
Amount               0
Transaction Type     0
Category             0
Account Name         0
Month                0
Original_Currency    0
Amount_LKR           0
dtype: int64


In [16]:
print("\n--- Sample ---")
print(df1.head())


--- Sample ---
        Date          Description   Amount Transaction Type  \
0 2018-01-01               Amazon    11.11            debit   
1 2018-01-02     Mortgage Payment  1247.44            debit   
2 2018-01-02      Thai Restaurant    24.22            debit   
3 2018-01-03  Credit Card Payment  2298.09           credit   
4 2018-01-04              Netflix    11.76            debit   

              Category   Account Name    Month Original_Currency  Amount_LKR  
0             Shopping  Platinum Card  2018-01               USD     3677.95  
1      Mortgage & Rent       Checking  2018-01               USD   412962.77  
2          Restaurants    Silver Card  2018-01               USD     8017.99  
3  Credit Card Payment  Platinum Card  2018-01               USD   760778.56  
4        Movies & Dvds  Platinum Card  2018-01               USD     3893.13  


# Save the cleaned dataset back to Google Drive

In [17]:
output_path = '/content/drive/MyDrive/FYP Data Cleaning/Clean Data/personal_transactions_cleaned.csv'
df1.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/FYP Data Cleaning/Clean Data/personal_transactions_cleaned.csv
